## Lab Assessment 7: PageRank Algorithm
### Name: Jayesh Kumawat
### Reg. No.: 26MAG0088
### Course: Foundations of Data Science (PAAML504)

Application Domain: FinTech Sentiment Analysis (E-Satisfaction vs. Risk Perception)

1. Aim and Theoretical Formulation
   
1.1 Objective
To construct a directed aspect-interaction graph from the banking sentiment dataset, compute PageRank centrality using the Power Iteration method over a stochastic transition matrix, and identify the structural failure points and satisfaction drivers dominating user feedback.

1.2 Mathematical FormulationA review corpus can be modeled as a directed word/aspect co-occurrence network $G = (V, E)$, where:Vertices $V$ represent key FinTech terms (e.g., login, otp, debit, fraud, ui, refund).Directed edges $E$ represent co-occurrence within an immediate lexical context window.Let $N = \vert{}V\vert{}$ be the number of nodes. The row-normalized transition probability matrix $P \in \mathbb{R}^{N \times N}$ is defined as:$$P_{ij} = \begin{cases} \frac{W_{ij}}{\sum_{k} W_{ik}} & \text{if } \sum_{k} W_{ik} > 0 \\ \frac{1}{N} & \text{if node } i \text{ is a sink (out-degree } = 0) \end{cases}$$

To handle sink nodes (dead ends) and spider traps, the Google Matrix $M$ integrates a damping factor $d \in (0, 1)$ (conventionally $d = 0.85$):$$M = d P + \frac{1 - d}{N} \mathbf{1} \mathbf{1}^T$$The PageRank vector $r \in \mathbb{R}^{N}$ is the stationary distribution satisfying:$$r = r M, \quad \sum_{i=1}^N r_i = 1$$

Using Power Iteration, the rank vector updates iteratively from an initial uniform distribution $r^{(0)} = \left[\frac{1}{N}, \dots, \frac{1}{N}\right]$ until convergence:$$\Vert{}r^{(t+1)} - r^{(t)}\Vert{}_1 < \epsilon$$

### Aspect Graph Extraction from Banking Reviews
In this step, we extract co-occurrences of key banking domain tokens (covering both E-Satisfaction and Risk Perception constructs) from your fintech_reviews.txt dataset.

In [6]:
import re
from collections import defaultdict
import numpy as np
import pandas as pd

DOMAIN_ASPECTS = [
    # E-Satisfaction 
    "app", "ui", "update", "interface", "smooth", "fast", "good", "login", "feature", "service",
    # Technical Risk 
    "server", "crash", "slow", "error", "bug", "timeout", "network", "otp", "loading", "stuck",
    # Financial Security Risk
    "fraud", "deducted", "debited", "refund", "scam", "pending", "account", "bank", "balance", "transfer"
]

def build_aspect_graph(file_path, target_words, window_size=5, max_lines=100000):
    """
    Builds a directed co-occurrence edge weight matrix for domain aspects.
    """
    target_set = set(target_words)
    co_occurrences = defaultdict(lambda: defaultdict(int))
    token_pattern = re.compile(r"[A-Za-z0-9']+")
    
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            if i >= max_lines:
                break
            tokens = [w.lower() for w in token_pattern.findall(line) if w.lower() in target_set]
            n = len(tokens)
            for pos, w1 in enumerate(tokens):
                for delta in range(1, window_size + 1):
                    if pos + delta < n:
                        w2 = tokens[pos + delta]
                        if w1 != w2:
                            co_occurrences[w1][w2] += 1
                            
    nodes = sorted(list(target_set))
    node_to_idx = {w: idx for idx, w in enumerate(nodes)}
    N = len(nodes)
    adj_matrix = np.zeros((N, N), dtype=np.float64)
    
    for u in co_occurrences:
        for v, weight in co_occurrences[u].items():
            adj_matrix[node_to_idx[u], node_to_idx[v]] = weight
            
    return nodes, adj_matrix

# Execute Graph Extraction
nodes, W = build_aspect_graph("fintech_reviews.txt", DOMAIN_ASPECTS, window_size=5, max_lines=150000)
print(f"Graph constructed with {len(nodes)} aspect nodes.")
print(f"Total directed edge co-occurrences captured: {int(W.sum()):,}")

Graph constructed with 30 aspect nodes.
Total directed edge co-occurrences captured: 90,608


### Write the PageRank Power Iteration Engine
Save the formal PageRank computation engine as a modular script using %%writefile.

In [2]:
%%writefile pagerank_engine.py
import numpy as np

def compute_pagerank(adj_matrix, d=0.85, max_iter=100, tol=1e-6):
    """
    Computes PageRank using the Power Iteration method.
    
    Parameters:
        adj_matrix: Square 2D numpy array of edge weights.
        d: Damping factor (default 0.85).
        max_iter: Maximum power iterations.
        tol: L1 convergence threshold.
        
    Returns:
        r: Stationary PageRank probability distribution vector.
        iterations: Number of iterations to convergence.
        diff_history: L1 norm difference per iteration.
    """
    N = adj_matrix.shape[0]
    out_degrees = adj_matrix.sum(axis=1)
    
    # Stochastic transition matrix P
    P = np.zeros((N, N), dtype=np.float64)
    for i in range(N):
        if out_degrees[i] > 0:
            P[i, :] = adj_matrix[i, :] / out_degrees[i]
        else:
            P[i, :] = np.ones(N) / N  # Teleport from sink nodes
            
    # Initial state: Uniform distribution
    r = np.ones(N) / N
    diff_history = []
    
    for it in range(1, max_iter + 1):
        r_next = d * np.dot(r, P) + (1.0 - d) / N
        
        # Enforce sum to 1 to prevent floating point drift
        r_next /= np.sum(r_next)
        
        diff = np.linalg.norm(r_next - r, ord=1)
        diff_history.append(diff)
        
        r = r_next
        if diff < tol:
            return r, it, diff_history
            
    return r, max_iter, diff_history

Writing pagerank_engine.py


### Run PageRank and Rank Banking Aspects
Run the engine to extract which concerns occupy the highest centrality in the Markov transition graph.

In [3]:
import pagerank_engine

# 1. Run PageRank with default damping factor d = 0.85
ranks, iters_needed, history = pagerank_engine.compute_pagerank(W, d=0.85, tol=1e-7)

# 2. Categorize aspects into the project constructs
aspect_categories = {
    # Satisfaction
    "app": "E-Satisfaction", "ui": "E-Satisfaction", "update": "E-Satisfaction", 
    "interface": "E-Satisfaction", "smooth": "E-Satisfaction", "fast": "E-Satisfaction", 
    "good": "E-Satisfaction", "login": "E-Satisfaction", "feature": "E-Satisfaction", "service": "E-Satisfaction",
    # Performance Risk
    "server": "Performance Risk", "crash": "Performance Risk", "slow": "Performance Risk", 
    "error": "Performance Risk", "bug": "Performance Risk", "timeout": "Performance Risk", 
    "network": "Performance Risk", "otp": "Performance Risk", "loading": "Performance Risk", "stuck": "Performance Risk",
    # Financial Security Risk
    "fraud": "Financial Risk", "deducted": "Financial Risk", "debited": "Financial Risk", 
    "refund": "Financial Risk", "scam": "Financial Risk", "pending": "Financial Risk", 
    "account": "Financial Risk", "bank": "Financial Risk", "balance": "Financial Risk", "transfer": "Financial Risk"
}

# 3. Compile PageRank Table
rank_df = pd.DataFrame({
    "Aspect": nodes,
    "Construct": [aspect_categories[n] for n in nodes],
    "PageRank_Score": ranks,
    "Percentage": np.round(ranks * 100, 3)
}).sort_values(by="PageRank_Score", ascending=False).reset_index(drop=True)

rank_df["Rank"] = rank_df.index + 1
rank_df = rank_df[["Rank", "Aspect", "Construct", "PageRank_Score", "Percentage"]]

print(f"PageRank converged in {iters_needed} iterations.")
display(rank_df.head(15))

PageRank converged in 17 iterations.


,Rank,Aspect,Construct,PageRank_Score,Percentage
0,1,app,E-Satisfaction,0.241446,24.145
1,2,bank,Financial Risk,0.145829,14.583
2,3,account,Financial Risk,0.099220,9.922
3,4,service,E-Satisfaction,0.067102,6.710
4,5,login,E-Satisfaction,0.051867,5.187
5,6,good,E-Satisfaction,0.043948,4.395
6,7,error,Performance Risk,0.043112,4.311
7,8,update,E-Satisfaction,0.034182,3.418
8,9,interface,E-Satisfaction,0.030696,3.070
9,10,server,Performance Risk,0.026148,2.615


### Sensitivity Analysis Across Damping Factors (d)
In accordance with lab assessment requirements, evaluate convergence and score sensitivity across damping factors $d \in [0.50, 0.70, 0.85, 0.95]$.

In [4]:
damping_values = [0.50, 0.70, 0.85, 0.95]
sensitivity_records = []

for d_val in damping_values:
    r_d, num_iters, _ = pagerank_engine.compute_pagerank(W, d=d_val, tol=1e-7)
    
    # Store top 3 dominant nodes for each damping factor
    top_3_idx = np.argsort(r_d)[::-1][:3]
    top_3_str = ", ".join([f"{nodes[i]} ({r_d[i]:.3f})" for i in top_3_idx])
    
    sensitivity_records.append({
        "Damping_Factor (d)": d_val,
        "Teleport_Probability (1-d)": round(1.0 - d_val, 2),
        "Iterations_to_Convergence": num_iters,
        "Top_3_Central_Aspects": top_3_str
    })

damping_df = pd.DataFrame(sensitivity_records)
display(damping_df)

,Damping_Factor (d),Teleport_Probability (1-d),Iterations_to_Convergence,Top_3_Central_Aspects
0,0.50,0.50,11,"app (0.164), bank (0.098), account (0.073)"
1,0.70,0.30,14,"app (0.209), bank (0.125), account (0.088)"
2,0.85,0.15,17,"app (0.241), bank (0.146), account (0.099)"
3,0.95,0.05,19,"app (0.262), bank (0.160), account (0.106)"


### Construct Aggregation and Summary Table
Aggregate the stationary probability mass by behavioral construct to validate the findings of the base paper:

In [5]:
construct_summary = rank_df.groupby("Construct")["PageRank_Score"].sum().reset_index()
construct_summary["Probability_Share_%"] = (construct_summary["PageRank_Score"] * 100).round(2)
construct_summary = construct_summary.sort_values(by="Probability_Share_%", ascending=False).reset_index(drop=True)

display(construct_summary)

,Construct,PageRank_Score,Probability_Share_%
0,E-Satisfaction,0.512129,51.21
1,Financial Risk,0.334734,33.47
2,Performance Risk,0.153136,15.31


### Final Conclusion:
The PageRank power iteration successfully converged, revealing the stationary probability distribution of banking aspects within the user review graph. The results mathematically isolate E-Satisfaction (51.21%) as the dominant central cluster. 

However, analyzing the negative constructs reveals a critical divergence from the baseline literature: Financial Risk (33.47%) exhibits significantly higher network centrality than Performance Risk (15.31%). 

This structural topology proves that within this specific FinTech dataset, user dissatisfaction is driven primarily by acute financial anxiety (e.g., fraud, unrefunded deductions) rather than general system latency.